In [3]:
import os
import openai

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


In [4]:
client = Client()

In [5]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [6]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('ca46b710-d05e-41d9-9fa4-c1bfb55c06f7'), created_at=datetime.datetime(2026, 8, 3, 8, 54, 7, 50283, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 8, 3, 8, 54, 7, 50283, tzinfo=TzInfo(0)), example_count=70, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.4.1-arm64-arm-64bit', 'sdk_version': '0.10.15', 'runtime_version': '3.11.15', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [12]:
list(client.list_examples(dataset_id=dataset.id,limit=10))[0].outputs

{'ground_truth': 'The price is not provided in the available chunks, so I cannot determine the current price.',
 'reference_context_ids': [],
 'reference_descriptions': []}

In [13]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs

### RAG Pipeline

In [16]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query,qdrant_client, k=5) :

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points : 
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)
    
    return {
        "retrieved_context_ids" : retrieved_context_ids,
        "retrieved_context" : retrieved_context,
        "retrieved_context_ratings" : retrieved_context_ratings,
        "similarity_scores" : similarity_scores
    }

def process_context(context):

    formatted_context = ""

    for id, chunk , rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context['retrieved_context_ratings']):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions :
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

def generate_answer(prompt) :

    response = openai.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[{"role":"system", "content": prompt}],
        reasoning_effort="low"
    )

    return response.choices[0].message.content

def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result



In [17]:
rag_pipeline("Can i get some charger?")

{'answer': 'Yes. Available chargers include:\n\n- **PWR+ 65W Laptop Charger for HP** — compatible with many HP Pavilion, Envy, Spectre, Stream, Chromebook, ProBook, and EliteBook models. It has a 12-foot cord and a 24-month exchange warranty.\n- **NetDot Magnetic Charging Cable** — 5-foot nylon-braided cables, 3-pack, compatible with USB-C and Micro USB devices. Supports 9V/2A fast charging.\n- **Anker Roav SmartCharge F3** — car Bluetooth FM transmitter with a Quick Charge 3.0 charging port.\n- **AHRISE Smart Power Strip** — includes four USB charging ports with 24W total output, plus six outlets.',
 'question': 'Can i get some charger?',
 'retrieved_context_ids': ['B005C53NHK',
  'B07JZ3NB7G',
  'B07DFBHDBT',
  'B07VV2MGMC',
  'B08P3HHBTS'],
 'retrieved_context': ['PWR+ 65W Laptop Charger for HP 741727-001 710412-001 HP Spectre X360 Stream 11 13 14 Pavilion Envy X360 Touchsmart 15 13 M6 Probook Elitebook Chromebook 11 G4 G5 EE - UL Listed AC Adapter Power Cord FEATURES / POWER SPECS 

### RAGAS metrics

In [18]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy


/var/folders/rj/bfp9n6495tbbdhxhbwlkkdn80000gn/T/ipykernel_70356/581455323.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/rj/bfp9n6495tbbdhxhbwlkkdn80000gn/T/ipykernel_70356/581455323.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/rj/bfp9n6495tbbdhxhbwlkkdn80000gn/T/ipykernel_70356/581455323.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecat

In [25]:
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-5.6-luna"),
    bypass_temperature=True,
    bypass_n=True,
)
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/rj/bfp9n6495tbbdhxhbwlkkdn80000gn/T/ipykernel_70356/3637829393.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
/var/folders/rj/bfp9n6495tbbdhxhbwlkkdn80000gn/T/ipykernel_70356/3637829393.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [21]:
result = rag_pipeline(reference_input["question"])

In [26]:
async def ragas_faithfulness(run, example) :

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


In [27]:
await ragas_faithfulness(result, "")

1.0

In [28]:
async def ragas_response_relevancy(run, example) : 

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [29]:
await ragas_response_relevancy(result,"")

np.float64(0.9161423199937779)

In [30]:
async def ragas_context_precision_id_based(run, example) :

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [31]:
await ragas_context_precision_id_based(result, reference_output)

0.0

In [32]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_output)